# Operations on data with stat_tool

## This notebook illustates how to perform basic operations on data with stat_tool.data_transform

The main data structures in stat_tool are Vectors and Histogram. Histogram is basically a 1-D array or a multiset of integers. Possibilities are offered of combining histograms with Merge, adding constants to values with Shift, filtering values with ValueSelect and renaming values with Transcode.    
The specificity of Vectors are the presence of several variables: the additional possible operations with respect to Histogram are merging  variables .

## Transform data on Vectors

We first load a data set (chene_sessile.vec)    

In [1]:
from openalea.stat_tool import (get_shared_data,
                                Histogram
)
from openalea.stat_tool.data_transform import (SelectVariable, 
                                               SelectIndividual, 
                                               Merge,
                                               MergeVariable,
                                               Shift,
                                               ValueSelect
)
from openalea.stat_tool.output import Plot
from openalea.stat_tool.vectors import Vectors
from openalea.stat_tool.cluster import Transcode

vec = Vectors(get_shared_data("chene_sessile.vec"))

In [2]:
print(vec)

Note that vec2 contains 6 variables.    

### Selecting variables

We can select the first two variables using function SelectVariable:

In [3]:
vec2 = SelectVariable(vec, [1,2], "Keep")

This is equivalent to using the method Vectors.select_variable, except that the last argument is Keep in [True, False] instead of Mode in ["Keep", "Reject"].

In [4]:
assert(str(vec2) == str(vec.select_variable([1,2], True)))

We can check that only 2 variables are remaining:

In [5]:
vec2.nb_variable

Setting argument "Keep" to False would discard variables [1,2]:

In [6]:
assert(vec.select_variable([1,2], False).nb_variable == 4)

### Selecting individuals

In the same way we can select variables, we can also select individuals:

In [7]:
subvec = SelectIndividual(vec,list(range(30, 120)), "Keep")

...which is equivalent to 

In [8]:
assert(str(subvec) == str(vec.select_individual(list(range(30, 120)), True)))

### Merging Vectors

Vectors with the same dimension can be merged (individual-wise):

In [9]:
vec_i1 = Vectors([[0, 0, 0], [1, 1, 1]])
vec_i2 = Vectors([[2, 2, 2], [3, 3, 3]])
vec_i3 = Vectors([[4, 4, 4]])
vec_merged = Merge(vec_i1, vec_i2, vec_i3)
vec_merged.nb_vector

This is equivalent to using the method Vectors.merge:

In [10]:
str(vec_i1.merge([vec_i2, vec_i3])) == str(vec_merged)

In [11]:
print(vec_merged[4])

It can be seen that the last vector from the merge Vectors is the last vector from vec_i3.

Similarly, Vectors with the same numbers of individuals can be merged (variable-wise):

In [12]:
vec_v2 = Vectors([[5], [5], [5], [5], [5]])
vec_vmerged = MergeVariable(vec_merged, vec_v2, RefSample=1)

The RefSample argument is optional here and is used only to give identifiers to vectors using one of the samples as a reference, here sample 1.

In [13]:
print(vec_vmerged[0])

It can be seen that the variable from vec_v2 has been added to the 3 variables from vec_merged.

This is equivalent to using the method Vectors.merge_variable:

In [14]:
print((vec_merged.merge_variable([vec_v2],1))[0])

### Shifting Vectors

Shifting a vector is the operation consisting in adding a constant to each value of a variable:

In [15]:
print(vec2)

In [16]:
print(Shift(vec2, 1, 2))

Here, variable 1 (second argument) has been shifted by 2 (third argument).    
If a Vectors object has only one variable, the variable does not have to be passed as an argument:

In [17]:
Shift(vec_v2, 2)[0]

As usual, it is equivalent to use method Vectors.shift:

In [18]:
print([vec2.get_min_value(0), vec2.shift(1, 2).get_min_value(0)])

Vectors can be shifted by negative values:

In [19]:
Shift(vec_i2, 1, -3)[0]

### Selecting individuals by values (filtering individuals)

Individual vectors in Vectors can be selected or discarded according to their values.   
For example,

In [20]:
ValueSelect(vec_merged, 1, 2, 3).nb_vector

selects all vectors in vec_merged such that variable 1 is between 2 and 3.   
To discard all such vectors, use argument 'Mode = "Reject"':

In [21]:
ValueSelect(vec_merged, 1, 2, 3, Mode="Reject").nb_vector

The object-oriented syntax is:

In [22]:
vec_merged.value_select(1, 2, 3, True)

where the last argument corresponds to 'Keep=True'.

### Transcoding Vectors

Transcoding the values of a Vectors object consists in replacing a value by another one. To use function Transcode, the list of values that replace the former ones is placed as argument:

In [23]:
Transcode(vec_merged, 1, [0, -1, -2, -3, -4])

This is equivalent to using method Vectors.transcode:

In [24]:
print([vec_merged.transcode(1, [0, -1, -2, -3, -4])[i][0] for i in range(vec_merged.nb_vector)])

In [25]:
print([vec_merged[i][0] for i in range(vec_merged.nb_vector)])

Note that 0 has been replaced by 0, 1 by -1, etc.

## Transform data on Histograms

### Merging histograms

Histogram objects can be merged, which consists in adding counts of each histogram.   
It can be seen from y-axis in the last figure that counts match the sum of counts of the 2 previous figures.

In [26]:
meri1 = Histogram(get_shared_data("meri1.his"))
meri2 = Histogram(get_shared_data("meri2.his"))
meri12 = Merge(meri1, meri2)

In [27]:
Plot(meri1)
Plot(meri2) 
Plot(meri12)

Arbitrary numbers of Histogram objects can be merged:

In [28]:
meri3 = Histogram(get_shared_data("meri3.his"))
meri123 = Merge(meri1, meri2, meri3)

This is equivalent to the method Histogram.merge:

In [29]:
meri123b = meri1.merge([meri2, meri3])
assert(str(meri123) == str(meri123b))

### Shifting histograms

As Vectors, Histogram objects can be shifted:

In [30]:
Plot(meri1)
Plot(Shift(meri1,3))

All values have been shifted by 3 (shift of the x-axis). This is equivalent to Histogram.shift:

In [31]:
assert(str(Shift(meri1,3)) == str(meri1.shift(3)))

### Selecting individuals by values (filtering histograms)

As Vectors, Histogram objects can be filtered:

In [32]:
Plot(meri1)
Plot(ValueSelect(meri1,15,25))

Here, only the values between 15 and 25 have been selected. This is equivalent with Histogram.value_select

In [33]:
assert(str(ValueSelect(meri1,15,25)) == str(meri1.value_select(15,25,True)))

### Transcoding Histograms

As Vectors, the values of Histogram objects can be transcoded.   
Note that:   
 * The first successive values in the range of the Histogram with null frequencies (range(0, Histogram.offset-1)) do not count, they cannot be transcoded.
 * The other values with null frequencies do count, they are include in the list defining transcoding.  

Here, to simplify the example, we restrict the Histogram to values 6, 7, 8, 9, 10.

In [34]:
meri_vselect = ValueSelect(meri1,6,10)
print(meri_vselect.display(Detail=2))

In [35]:
range(0, meri_vselect.offset-1)

The transcoding is
 * $6 \rightarrow 2$ 
 * $7 \rightarrow 4$ 
 * $8 \rightarrow 1$ 
 * $9 \rightarrow 3$ 
 * $10 \rightarrow 0$ 

In [36]:
meri_transcoded = Transcode(meri_vselect,[2, 4, 1, 3, 0])
print(meri_transcoded.display(Detail=2))